# Train test split + Baseline model

1. Filter out movies with fewer than 5 ratings (decided in notebook 01: it drops ~60% of movies but only ~1.3% of ratings).
2. Re-index users and movies to contiguous integers (matrix models need `0..n-1` indices).
3. Split **per user by time**: each user's oldest ratings → train, next → validation, newest → test.
4. Score three non-learning baselines (random, popularity, damped mean rating) with ranking metrics.

#### Why baselines first
Popularity is a strong baseline here because the top 10% of movies hold ~88% of all ratings.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

D   = Path("../data/interim/sample_10pct")
OUT = Path("../data/processed/sample_10pct")
OUT.mkdir(parents=True, exist_ok=True)

# Decided from notebook 1

MIN_MOVIE_RATINGS = 5      # drop movies rated fewer times than this
MIN_USER_RATINGS  = 10     # safety net after movie filtering
REL_THRESHOLD     = 4.0    # a held-out rating >= this counts as "relevant"
HOLD_FRAC, HOLD_MIN, HOLD_MAX = 0.10, 2, 10   # per-user val/test size = 10% of history, clipped to [2, 10]
K    = 10
SEED = 42

ratings = pd.read_parquet(D / "ratings.parquet")
print(ratings.shape)

(3237353, 4)


## 1 · Filter rarely-rated movies
**Why:** a movie with 1 to 4 ratings gives collaborative filtering almost nothing to learn from. They stay in the catalogue and will be handled by content features later; we just don't train on them or evaluate against them.
**Check:** you should see ~17,241 movies and ~98.7% of ratings kept, matching the threshold table in notebook 01.

In [2]:
n0 = len(ratings)
movie_cnt = ratings.groupby("movieId")["movieId"].transform("size")
ratings = ratings[movie_cnt >= MIN_MOVIE_RATINGS].copy()

user_cnt = ratings.groupby("userId")["userId"].transform("size")
n_users_dropped = (ratings.loc[user_cnt < MIN_USER_RATINGS, "userId"]).nunique()
ratings = ratings[user_cnt >= MIN_USER_RATINGS].copy()

print(f"ratings kept: {len(ratings):,} ({len(ratings)/n0:.2%})")
print(f"movies kept:  {ratings.movieId.nunique():,}")
print(f"users kept:   {ratings.userId.nunique():,} (dropped {n_users_dropped})")

ratings kept: 3,194,074 (98.66%)
movies kept:  17,241
users kept:   20,093 (dropped 1)


## 2 · Re-index users and movies
**Why:** MovieLens IDs are sparse (movie IDs run into the 100,000s). Matrix factorization stores one vector per user and per movie in an array, so we need contiguous indices `0..n-1`. We keep the mappings so we can translate back to titles.

In [3]:
user_ids = np.sort(ratings.userId.unique())
item_ids = np.sort(ratings.movieId.unique())

u_map = pd.Series(np.arange(len(user_ids), dtype="int32"), index=user_ids)
i_map = pd.Series(np.arange(len(item_ids), dtype="int32"), index=item_ids)

ratings["u"] = ratings["userId"].map(u_map).astype("int32")
ratings["i"] = ratings["movieId"].map(i_map).astype("int32")

n_users, n_items = len(user_ids), len(item_ids)
print(n_users, "users |", n_items, "items")

20093 users | 17241 items


## 3 · Per-user temporal split
**Why:** a random split lets the model train on a user's future to predict their past (leakage). Instead, for every user we sort their ratings by time and hold out their **most recent** ones.

- Test = the last `n_hold` ratings, validation = the `n_hold` before that, train = everything older.
- `n_hold = 10%` of the user's history, clipped to between 2 and 10, so light users still get a usable holdout and heavy users don't dominate the metrics.
- Ties in timestamp (users often rate many movies in one sitting) are broken by item index, so the split is deterministic.

**Caveat:** this is a *per-user* temporal split, not a global cut-off date. A test rating from user A may predate a train rating from user B. It's the standard compromise; a global time split is stricter and could be a later robustness check.

In [4]:
ratings = ratings.sort_values(["u", "timestamp", "i"]).reset_index(drop=True)

g = ratings.groupby("u")
n_user         = g["u"].transform("size")
pos_from_end   = g.cumcount(ascending=False)          # 0 = the user's newest rating
n_hold = np.clip(np.round(HOLD_FRAC * n_user), HOLD_MIN, HOLD_MAX).astype(int)

ratings["split"] = np.where(pos_from_end < n_hold, "test",
                   np.where(pos_from_end < 2 * n_hold, "val", "train"))

print(ratings["split"].value_counts())

split
train    2919680
val       137197
test      137197
Name: count, dtype: int64


## 4 · Sanity-check the split
Never trust a split without testing it. We verify (a) time order holds for every user, and (b) how many held-out interactions involve a movie that never appears in train (a "cold" target no collaborative model can score).

In [5]:
t = ratings.groupby(["u", "split"])["timestamp"].agg(["min", "max"]).unstack("split")
assert (t[("max", "train")] <= t[("min", "val")]).all()
assert (t[("max", "val")]   <= t[("min", "test")]).all()
print("time order OK for all users")

ties = (t[("max", "train")] == t[("min", "val")]).mean()
print(f"users whose train/val boundary falls on identical timestamps: {ties:.1%}")

train = ratings[ratings.split == "train"]
val   = ratings[ratings.split == "val"]
test  = ratings[ratings.split == "test"]

train_items = set(train.i)
for name, d in (("val", val), ("test", test)):
    print(f"{name}: {len(d):,} ratings | on items with no train ratings: {(~d.i.isin(train_items)).mean():.2%}")

time order OK for all users
users whose train/val boundary falls on identical timestamps: 17.0%
val: 137,197 ratings | on items with no train ratings: 0.00%
test: 137,197 ratings | on items with no train ratings: 0.02%


In [6]:
cols = ["u", "i", "rating", "timestamp"]
train[cols].to_parquet(OUT / "train.parquet", index=False)
val[cols].to_parquet(OUT / "val.parquet", index=False)
test[cols].to_parquet(OUT / "test.parquet", index=False)
pd.DataFrame({"u": u_map.values, "userId": u_map.index}).to_parquet(OUT / "user_map.parquet", index=False)
pd.DataFrame({"i": i_map.values, "movieId": i_map.index}).to_parquet(OUT / "item_map.parquet", index=False)

## 5 · Evaluation machinery
**Relevant item** = a held-out movie the user rated ≥ 4.0.
**Candidates** = every movie *except* those the user already has in their history. For validation we exclude train items; for test we'd exclude train + val, since by then the user has seen those.
**Metrics @10:** Recall (what fraction of their liked held-out movies did we find?), NDCG (did we find them near the top?), HitRate (at least one?), plus **coverage** (what share of the catalogue ever gets recommended; popularity will be tiny).

In [7]:
def to_sets(df):
    return df.groupby("u")["i"].agg(lambda s: set(s.tolist())).to_dict()

seen_train = to_sets(train)
rel_val    = to_sets(val[val.rating >= REL_THRESHOLD])
print(f"validation users with >=1 relevant item: {len(rel_val):,} of {n_users:,}")
print(f"mean relevant items per user: {np.mean([len(s) for s in rel_val.values()]):.2f}")


def topk_from_order(order, seen_by_user, users, k=K):
    """Same global ranking for everyone; skip anything the user has already seen."""
    ranked = {}
    for u in users:
        seen = seen_by_user.get(u, ())
        out = []
        for i in order:
            if i not in seen:
                out.append(i)
                if len(out) == k:
                    break
        ranked[u] = out
    return ranked


def evaluate_topk(ranked, relevant, k=K):
    discounts = 1 / np.log2(np.arange(2, k + 2))
    recalls, ndcgs, hits, shown = [], [], [], set()
    for u, rel in relevant.items():
        top = ranked[u][:k]
        shown.update(top)
        gains = np.array([m in rel for m in top], dtype=float)
        dcg  = (gains * discounts[:len(top)]).sum()
        idcg = discounts[:min(len(rel), k)].sum()
        recalls.append(gains.sum() / len(rel))
        ndcgs.append(dcg / idcg)
        hits.append(float(gains.sum() > 0))
    return {f"Recall@{k}": np.mean(recalls), f"NDCG@{k}": np.mean(ndcgs),
            f"HitRate@{k}": np.mean(hits), "Coverage": len(shown) / n_items}

validation users with >=1 relevant item: 18,392 of 20,093
mean relevant items per user: 4.10


## 6 · Three baselines
- **Random:** a shuffled catalogue. The floor: anything below this is a bug.
- **Popularity:** rank movies by number of train ratings. No personalisation at all, but strong because attention is so concentrated.
- **Damped mean rating:** rank by average rating, shrunk toward the global mean so a movie with one 5-star rating doesn't win. Tests whether *quality* alone beats *popularity*.

None of these has anything to tune on validation except the damping constant, so they're not fitted to the test set. We'll compute test numbers for all models together at the end.

In [8]:
rng = np.random.default_rng(SEED)
users = list(rel_val.keys())

counts = np.bincount(train.i, minlength=n_items)
sums   = np.bincount(train.i, weights=train.rating, minlength=n_items)
mu, m  = train.rating.mean(), 20          # m = damping strength (pseudo-ratings at the global mean)

orders = {
    "random":       rng.permutation(n_items).tolist(),
    "popularity":   np.lexsort((np.arange(n_items), -counts)).tolist(),
    "damped_mean":  np.lexsort((np.arange(n_items), -((sums + m * mu) / (counts + m)))).tolist(),
}

results = {name: evaluate_topk(topk_from_order(o, seen_train, users), rel_val)
           for name, o in orders.items()}
pd.DataFrame(results).T.round(4)

,Recall@10,NDCG@10,HitRate@10,Coverage
random,0.0004,0.0003,0.0018,0.0009
popularity,0.0574,0.0415,0.1620,0.0137
damped_mean,0.0232,0.0135,0.0704,0.0042
